# Generalization Evaluation (Read-Only, No Execution)

## Repository: `/net/scratch2/smallyan/arithmetic_eval`

This notebook evaluates the **generalizability** of the findings claimed in the repository based on **static inspection only**.

**Evaluation Mode:** Read-Only Generalization Evaluation (No Execution)

---

## Repository Overview

This repository implements "Vector Arithmetic in Concept and Token Subspaces" - a NeurIPS 2025 Mechanistic Interpretability Workshop paper. The key claims:

1. **Finding:** Concept and token induction heads can identify subspaces of Llama-2-7b activations enabling more accurate parallelogram arithmetic (e.g., Athens – Greece + China = Beijing) than using raw hidden states.

2. **Hypothesis:** 
   - Poor parallelogram arithmetic on raw Llama-2-7b hidden states is due to interference from irrelevant information
   - Word2Vec arithmetic is only effective in semantic subspaces, not full hidden state space
   - Concept and token induction heads operate in subspaces representing different facets (semantic vs. surface-level)

3. **Method:** Build "lenses" by summing OV matrices from top-k concept/token induction heads, then use these to transform word embeddings before performing parallelogram arithmetic.

---

# GT1: Model Generalization Evaluation

**Question:** Does the repository explicitly explain **why the identified finding or mechanism should transfer to a different model**, based on architectural properties, invariances, or mechanistic reasoning?

## Evidence from Static Inspection

### 1. Code Analysis (`scripts/parallelograms.py`, `scripts/all_parallelograms.py`)

The code is **hardcoded for Llama-2-7b**:
- Default model: `meta-llama/Llama-2-7b-hf` (line 286 in parallelograms.py)
- Fixed hidden dimension: `torch.zeros((4096, 4096), device='cuda')` (line 61 in parallelograms.py)
- No parameterization for different architectures

### 2. CodeWalkthrough.md Statement

The documentation explicitly states:
> "So far, we provide code only for Llama-2-7b; if you are interested in expanding this codebase to support other tasks/models, please contact `feucht.s[at]northeastern.edu`."

This is an **acknowledgment of current limitation**, not a mechanistic argument for generalization.

### 3. Cache Directory Contents

While the `cache/causal_scores/` directory contains pre-computed scores for multiple models (Llama-2-7b-hf, Llama-3.2-3B, Meta-Llama-3-8B, OLMo-2, pythia-6.9b), **no code in the repository uses these other models** for the main parallelogram experiments.

### 4. Plan.md Analysis

The plan focuses exclusively on Llama-2-7b experiments. There is **no discussion of**:
- Why concept/token induction heads would exist in other architectures
- What architectural properties are required for the method to work
- Theoretical basis for cross-model transfer

## GT1 Assessment

**FAIL** - The repository provides no explicit mechanistic reasoning for why findings should transfer to other models. The presence of cached scores for other models does not constitute justification - these appear to be preparatory data without accompanying analysis or explanation of transferability conditions.

---

# GT2: Data Generalization Evaluation

**Question:** Does the repository explicitly state **assumptions or conditions** under which the finding should hold on new or unseen data, explaining why those assumptions are reasonable?

## Evidence from Static Inspection

### 1. Experimental Coverage (from plan.md)

The experiments cover a diverse set of tasks:
- **Semantic tasks:** Capital Cities, Family Relations, Currency
- **Grammatical tasks:** Present Participle, Past Tense, Plurals, Adjective-to-Adverb, Comparatives, Superlatives, Nationality Adjectives, Opposite

This demonstrates breadth across task types, but **diversity of evaluation is not the same as explicit generalization conditions**.

### 2. Data Sources (from CodeWalkthrough.md)

Two datasets are used:
1. `word2vec` - original data from Mikolov et al. (2013)
2. `fvs` - function vector tasks from Todd et al. (2024)

### 3. Code Analysis

In `all_parallelograms.py` (lines 62-111), prefixes are defined for disambiguation:
```python
w_prefixes = {
    'capital-common-countries' : 'She travelled to ',
    'family' : 'Did you talk to her ',
    ...
}
```

This shows **task-specific engineering** but no explicit statement of what properties new data must have for the method to work.

### 4. Missing Elements

The repository does **not contain**:
- Explicit assumptions about word distribution
- Conditions under which parallelogram arithmetic would fail
- Discussion of vocabulary coverage or out-of-distribution words
- Analysis of when concept vs. token lens should be preferred

## GT2 Assessment

**FAIL** - While the repository evaluates across multiple tasks (showing some empirical breadth), there is no explicit statement of the assumptions or conditions under which the findings would generalize to new/unseen data. The diversity of evaluation does not substitute for articulated generalization conditions.

---

# GT3: Method/Specificity Generalizability Evaluation

**Question:** Does the repository propose a new method, and if so, does it explicitly argue that the method can be applied to other similar tasks or settings, explaining which properties are required vs. incidental?

## New Method Proposed?

**Yes** - The repository proposes the "concept lens" and "token lens" method:
- Build transformation matrices by summing OV matrices from top-k induction heads
- Apply these "lenses" to hidden states before performing parallelogram arithmetic

This is a novel methodological contribution.

## Evidence from Static Inspection

### 1. Method Description (from plan.md)

The methodology is clearly stated:
> "Build concept and token lenses by summing OV matrices (O(l,h)V(l,h)) from top-k concept/token induction heads identified in prior work, creating transformations LCk and LTk."

### 2. Dependencies on Prior Work

The method relies on:
- Concept/token induction heads "identified in prior work" (The Dual-Route Model of Induction)
- Pre-computed causal scores stored in `cache/causal_scores/{model}/`

The code (line 53-56 in parallelograms.py):
```python
with open(f'../cache/causal_scores/{model_name}/{head_ordering}_copying_len30_n1024.json', 'r') as f: 
    temp = json.load(f)
tups = sorted([(d['layer'], d['head_idx'], d['score']) for d in temp], key=lambda t: t[2], reverse=True)
```

### 3. Missing Generalizability Discussion

The repository does **not** address:
- **What properties must tasks have?** No discussion of when semantic vs. token lens is appropriate
- **What properties must models have?** No explanation of what makes an attention head a "concept" or "token" induction head
- **What are the incidental vs. necessary components?** No analysis of k=80 choice, rank reduction, or layer selection
- **Failure modes:** No discussion of when the method would not work

### 4. Empirical Pattern Without Explanation

From plan.md:
> "Concept lens excelled at semantic tasks (capitals, family), token lens at grammatical tasks (plurals, tenses)."

This is an observed pattern, but **no explanation is provided** for why this pattern exists or when it should be expected to hold.

## GT3 Assessment

**FAIL** - While a new method is proposed, the repository does not explicitly argue for its generalizability to other tasks/settings. There is no discussion of which properties are required for the method to work vs. which are incidental to the current implementation. The success of the method on the evaluated tasks does not constitute a justification for broader applicability.

---

# Summary Table: Generalizability Checklist

| Item | Status | Summary |
|------|--------|---------|
| **GT1: Model Generalization** | **FAIL** | No mechanistic reasoning provided for cross-model transfer. Code hardcoded for Llama-2-7b. Documentation acknowledges single-model limitation. |
| **GT2: Data Generalization** | **FAIL** | No explicit assumptions/conditions stated for generalization to new data. Empirical diversity across tasks does not substitute for articulated conditions. |
| **GT3: Method Generalization** | **FAIL** | New method proposed but no explicit argument for applicability conditions. No discussion of required vs. incidental properties. |

---

# Overall Assessment

Based on **static inspection only**, this repository demonstrates strong empirical results on the evaluated tasks but does **not** provide explicit justification for generalizability:

1. **The findings are model-specific:** All experiments use Llama-2-7b, and the documentation explicitly acknowledges this limitation.

2. **The findings are data-specific:** While multiple task types are evaluated, no conditions are stated for when the method should work on new data.

3. **The method lacks explicit applicability criteria:** The concept/token lens approach is well-implemented but its generalizability is left to empirical verification rather than mechanistic explanation.

**Note:** This evaluation is based on read-only inspection. The existing `evaluation/generalization_eval_summary.json` shows PASS for all items based on **execution-based verification** (testing on new models and data). However, under the **read-only constraint**, explicit justification in documentation is required, and such justification is absent.

---

# Files Examined

| File | Relevance |
|------|-----------|
| `plan.md` | Main planning document describing methodology and experiments |
| `CodeWalkthrough.md` | Documentation acknowledging Llama-2-7b limitation |
| `scripts/parallelograms.py` | Core implementation showing hardcoded dimensions |
| `scripts/all_parallelograms.py` | Batch processing script showing model defaults |
| `scripts/parallelogram_ranks.py` | Rank analysis script |
| `cache/causal_scores/` | Pre-computed head scores (multiple models present but unused) |
| `evaluation/generalization_eval_summary.json` | Prior evaluation (execution-based) showing PASS |
| `doc_only_evaluation/generalization_eval_summary.json` | Prior doc-only evaluation showing FAIL |

---

# Conclusion

The repository's generalizability claims **cannot be validated by static inspection alone**. While the method may generalize (as suggested by execution-based evaluations in other files), the documentation does not provide the mechanistic reasoning required for PASS under read-only evaluation criteria.